# Entrenar les bounding boxes a Colab

Detector de `depth_column`, `interpreted_lithology_column` i `lithology_legend`. Activa **Entorn d’execució → Canvia el tipus → GPU (T4 o superior)**.

Al teu ordinador executa `python3 tools/package_colab.py`; puja `artifacts/colab_bundle.zip` a **My Drive/subsurface-bboxes/** i obre aquest notebook a Colab. El ZIP inclou el codi i els TIFF anotats, sense el duplicat. Els retalls es preparen al disc local de Colab; els checkpoints i informes es guarden a Drive. Cal internet per instal·lar dependències i descarregar els pesos preentrenats; la predicció no utilitza APIs externes.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/subsurface-bboxes')
BUNDLE_PATH = DRIVE_ROOT / 'colab_bundle.zip'
WORKDIR = Path('/content/subsurface-bboxes')
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
print('Pesos i informes:', RUN_DIR)


## Extreure el projecte i instal·lar dependències

El dataset original ocupa aproximadament 1 GB. Els TIFF més grans poden utilitzar aproximadament 1,5 GB addicionals de disc temporal mentre libvips els descodifica.


In [ ]:
import zipfile
import subprocess
import sys
import os

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Puja colab_bundle.zip a {BUNDLE_PATH}')
WORKDIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    for member in bundle.infolist():
        target = (WORKDIR / member.filename).resolve()
        if not target.is_relative_to(WORKDIR.resolve()):
            raise ValueError(f'Ruta invàlida dins el ZIP: {member.filename}')
    bundle.extractall(WORKDIR)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import torch
import pyvips
import json

if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU al menú Entorn d’execució abans d’entrenar.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('libvips:', '.'.join(str(pyvips.version(i)) for i in range(3)))
DATASET = WORKDIR / 'dataset'
PREPARED = WORKDIR / 'artifacts' / 'dataset'


## Auditar i preparar els retalls

La separació per pou/família es fa **abans** dels retalls. S’exclouen les imatges sense anotació i el duplicat. Amb la llavor 42 s’obtenen 41 documents d’entrenament i 10 de validació. Cada document aporta com a màxim 16 retalls; l’amplada de treball és 1.536 px. L’auditoria no descodifica els píxels.


In [ ]:
from bbox_detector.data import prepare_dataset

audit = prepare_dataset(DATASET, PREPARED, materialize=False)
print(json.dumps(audit['summary'], indent=2))
print('Exclosos:', audit['excluded'])


In [ ]:
manifest = prepare_dataset(DATASET, PREPARED, imgsz=1536, max_tiles=16, seed=42)
print(json.dumps(manifest['summary'], indent=2))
assert manifest['materialized']
assert not ({r['group'] for r in manifest['records'] if r['split'] == 'train'} &
            {r['group'] for r in manifest['records'] if r['split'] == 'val'})


## Entrenar YOLO11s

60 èpoques, parada anticipada amb paciència 10 i batch inicial 4. Si falta memòria CUDA, el codi reintenta amb batch 2 i després 1. Els reintents comencen un entrenament nou i queden registrats. `best.pt` es tria segons les mètriques dels retalls de validació; després avaluarem les caixes reconstruïdes del document complet.


In [ ]:
from bbox_detector.training import train_model

training = train_model(
    PREPARED / 'dataset.yaml', RUN_DIR / 'training',
    model='yolo11s.pt', epochs=60, patience=10,
    imgsz=1536, batch=4, device='0', workers=2, seed=42,
)
BEST_WEIGHTS = Path(training['best_weights'])
print('Checkpoint:', BEST_WEIGHTS)
print('Temps d’entrenament (min):', round(training['elapsed_seconds'] / 60, 1))


## Validar documents complets

La predicció només rep la imatge. Les anotacions s’utilitzen després per mesurar IoU i errors de vores. Es guarden les deteccions, comparacions visuals, CSV i `report.json`. Les regions absents compten com a IoU zero. L’informe inclou el percentatge de documents amb **les tres caixes amb IoU ≥ 0,8**.

La família D582, que conté totes les imatges d’1 bit, queda sencera a entrenament amb aquest split. Aquest informe no mesura generalització a una nova família d’1 bit.


In [ ]:
from bbox_detector.evaluation import evaluate_manifest

report = evaluate_manifest(
    PREPARED / 'manifest.json', BEST_WEIGHTS, RUN_DIR / 'validation',
    split='val', device='0', imgsz=1536, conf=0.25,
)
print(json.dumps(report['summary'], indent=2))
print('Informes i comparacions:', RUN_DIR / 'validation')


## Predir una imatge i exportar els resultats

Pots canviar `IMAGE_PATH` per un TIFF nou, sense JSON d’anotació. El fitxer de sortida conserva coordenades de la imatge original i màxims exclusius. Si alguna regió no es resol després del reintent, el predictor retorna un error explícit.


In [ ]:
from bbox_detector.inference import predict_document, InferenceError
from bbox_detector.common import write_json

IMAGE_PATH = Path(next(r['image_path'] for r in manifest['records'] if r['split'] == 'val'))
try:
    prediction = predict_document(IMAGE_PATH, BEST_WEIGHTS, device='0')
except InferenceError as error:
    print('Imatge sense totes les regions resoltes:', error)
    write_json(RUN_DIR / 'example_prediction_error.json', {
        'image_name': IMAGE_PATH.name, 'error': str(error),
        'partial_boxes': error.partial_boxes, 'diagnostics': error.diagnostics,
    })
else:
    write_json(RUN_DIR / 'example_prediction.json', prediction)
    print(json.dumps(prediction, indent=2))


In [ ]:
import shutil

EXPORT = RUN_DIR / 'inference_export'
EXPORT.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_WEIGHTS, EXPORT / 'best.pt')
for filename in ['predict_bboxes.py', 'requirements.txt', 'README.md']:
    shutil.copy2(WORKDIR / filename, EXPORT / filename)
if (WORKDIR / 'build_manifest.json').is_file():
    shutil.copy2(WORKDIR / 'build_manifest.json', EXPORT / 'build_manifest.json')
shutil.copytree(WORKDIR / 'bbox_detector', EXPORT / 'bbox_detector', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy2(RUN_DIR / 'validation' / 'report.json', EXPORT / 'validation_report.json')
archive = shutil.make_archive(str(RUN_DIR / 'inference_export'), 'zip', EXPORT)
print('Paquet per executar localment:', archive)
print('Comanda: python predict_bboxes.py --image document.tif --weights best.pt --output document.json')
